In [1]:
# ── Transfer Learning Cell ───────────────────────────────────────────────────
# Stage 1: Load Semantic3D pretrained weights (encoder + decoder only)
# Stage 2: Fine-tune on Purdue dataset with new fc0 (D_IN=5) and fc1 (C=10)
# ─────────────────────────────────────────────────────────────────────────────
import os
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

import numpy as np
import time
import sys
from pathlib import Path
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.utils.tensorboard import SummaryWriter

torch.backends.cudnn.benchmark     = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.enabled       = True

if torch.cuda.is_available():
    torch.cuda.empty_cache()

RANDLA_ROOT = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch")
sys.path.insert(0, str(RANDLA_ROOT))
sys.path.insert(0, str(RANDLA_ROOT / "utils"))
os.chdir(RANDLA_ROOT)

from model import RandLANet
from utils.metrics import accuracy, intersection_over_union

try:
    from torch_points_kernels import knn as knn_fn
except ImportError:
    from torch_points import knn as knn_fn

# ── Parameters ────────────────────────────────────────────────────────────
DATA_DIR           = Path(r"/mnt/e/Mona/Courses/AI/Project/Checkpoint_1/RandLA-Net-pytorch/Dataset/Train")
LOGS_DIR           = RANDLA_ROOT / "runs" / "transfer_learning"
SEMANTIC3D_CKPT    = RANDLA_ROOT / "checkpoints" / "randlanet_semantic3d.pth"
RESUME_CKPT        = None    # set to a transfer .pth to resume, or None to start fresh

# Target domain (Purdue)
D_IN            = 5          # X Y Z Intensity PointSourceID
NUM_CLASSES     = 10         # Purdue classes
NUM_NEIGHBORS   = 16
DECIMATION      = 4
NUM_LAYERS      = 5
NUM_POINTS      = 20480
EPOCHS          = 50
SAVE_FREQ       = 10
VAL_SPLIT       = 0.2
BATCH_SIZE      = 1
NUM_WORKERS     = 0
SEP             = r"\s+"
XYZ_COLS        = [0, 1, 2]
FEAT_COLS       = [3, 4]     # Intensity, PointSourceID
LABEL_COL       = 5

# ── Two-stage learning rates ───────────────────────────────────────────────
# Lower LR for pretrained encoder/decoder (don't destroy learned features)
# Higher LR for new fc0 and fc1 (train from scratch)
LR_PRETRAINED   = 1e-4       # encoder + decoder + bottleneck + bn0
LR_NEW_LAYERS   = 1e-3       # fc0 (new D_IN) + fc1 (new NUM_CLASSES)
SCHEDULER_GAMMA = 0.95
# ──────────────────────────────────────────────────────────────────────────

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(f"Using device : {device}")
if device.type == "cuda":
    print(f"  GPU    : {torch.cuda.get_device_name(0)}")
    print(f"  Memory : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
LOGS_DIR.mkdir(parents=True, exist_ok=True)

# ── Move inputs dict to device ────────────────────────────────────────────
def inputs_to_device(inputs, device):
    return {
        'features':         inputs['features'].to(device),
        'coords':           [c.to(device) for c in inputs['coords']],
        'neighbor_indices': [n.to(device) for n in inputs['neighbor_indices']],
        'sub_idx':          [s.to(device) for s in inputs['sub_idx']],
        'interp_idx':       [i.to(device) for i in inputs['interp_idx']],
    }

# ── KNN input builder ─────────────────────────────────────────────────────
def build_inputs(pts, num_layers, num_neighbors, decimation):
    coords_list   = []
    neighbor_list = []
    sub_idx_list  = []
    interp_list   = []
    pc = pts[:, :3].copy()
    for i in range(num_layers):
        pc_tensor = torch.from_numpy(pc).unsqueeze(0)
        N_i       = pc.shape[0]
        neighbor_idx, _ = knn_fn(
            pc_tensor.contiguous(), pc_tensor.contiguous(), num_neighbors)
        N_sub  = N_i // decimation
        pool_i = neighbor_idx[:, :N_sub, :]
        pc_sub = pc[:N_sub, :]
        up_i, _ = knn_fn(
            torch.from_numpy(pc_sub).unsqueeze(0).contiguous(),
            pc_tensor.contiguous(), 1)
        coords_list.append(pc_tensor)
        neighbor_list.append(neighbor_idx.long())
        sub_idx_list.append(pool_i.long())
        interp_list.append(up_i.long())
        pc = pc_sub
    return {
        'features':         torch.from_numpy(pts).unsqueeze(0),
        'coords':           coords_list,
        'neighbor_indices': neighbor_list,
        'sub_idx':          sub_idx_list,
        'interp_idx':       interp_list,
    }

# ── Dataset ───────────────────────────────────────────────────────────────
class PointCloudDataset(Dataset):
    def __init__(self, files, num_points, num_layers, num_neighbors,
                 decimation, xyz_cols, feat_cols, label_col, sep):
        self.files         = files
        self.num_points    = num_points
        self.num_layers    = num_layers
        self.num_neighbors = num_neighbors
        self.decimation    = decimation
        self.xyz_cols      = xyz_cols
        self.feat_cols     = feat_cols
        self.label_col     = label_col
        self.sep           = sep

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        import pandas as pd
        df = pd.read_csv(self.files[idx], sep=self.sep,
                         header=None, engine="python")
        xyz    = df.iloc[:, self.xyz_cols].values.astype(np.float32)
        feat   = df.iloc[:, self.feat_cols].values.astype(np.float32)
        labels = df.iloc[:, self.label_col].values.astype(np.int64)
        f_min  = feat.min(axis=0, keepdims=True)
        f_max  = feat.max(axis=0, keepdims=True)
        feat   = (feat - f_min) / (f_max - f_min + 1e-8)
        pts    = np.hstack([xyz, feat]).astype(np.float32)
        N = len(pts)
        if N >= self.num_points:
            idx_s = np.random.choice(N, self.num_points, replace=False)
        else:
            idx_s = np.concatenate([np.arange(N),
                                    np.random.choice(N, self.num_points - N,
                                                     replace=True)])
        pts    = pts[idx_s]
        labels = np.clip(labels[idx_s], 0, NUM_CLASSES - 1)
        inputs = build_inputs(pts, self.num_layers,
                              self.num_neighbors, self.decimation)
        return inputs, torch.from_numpy(labels)

# ── Collate fn ────────────────────────────────────────────────────────────
def collate_fn(batch):
    inputs_list, labels_list = zip(*batch)
    labels = torch.stack(labels_list, dim=0)
    def stack_layers(tensor_lists):
        n = len(tensor_lists[0])
        return [torch.cat([tensor_lists[b][i]
                           for b in range(len(tensor_lists))], dim=0)
                for i in range(n)]
    inputs = {
        'features':         torch.cat([inp['features']         for inp in inputs_list], dim=0),
        'coords':           stack_layers([inp['coords']           for inp in inputs_list]),
        'neighbor_indices': stack_layers([inp['neighbor_indices'] for inp in inputs_list]),
        'sub_idx':          stack_layers([inp['sub_idx']          for inp in inputs_list]),
        'interp_idx':       stack_layers([inp['interp_idx']       for inp in inputs_list]),
    }
    return inputs, labels

# ── Compute class weights ────────────────────────────────────────────────
def compute_class_weights(files, num_classes, sep, label_col):
    import pandas as pd
    counts = np.zeros(num_classes, dtype=np.float64)

    print("\nComputing class weights from training files ...")
    for f in tqdm(files, desc="Class weights", leave=False):
        df = pd.read_csv(f, sep=sep, header=None, engine="python")
        labels = df.iloc[:, label_col].values.astype(np.int64)
        labels = labels[(labels >= 0) & (labels < num_classes)]
        binc = np.bincount(labels, minlength=num_classes)
        counts += binc

    print("Raw class counts:", counts.astype(np.int64).tolist())

    counts[counts == 0] = 1.0
    weights = 1.0 / np.log(1.2 + counts)
    weights = weights / weights.mean()

    return torch.tensor(weights, dtype=torch.float32).to(device)

# ── Train / val split ─────────────────────────────────────────────────────
all_files = sorted(DATA_DIR.glob("*.txt"))
print(f"\nTotal tile files found : {len(all_files)}")
if len(all_files) == 0:
    raise FileNotFoundError(f"No .txt files found in '{DATA_DIR}'")

n_val       = max(1, int(len(all_files) * VAL_SPLIT))
n_train     = len(all_files) - n_val
perm        = torch.randperm(len(all_files)).tolist()
train_files = [all_files[i] for i in perm[:n_train]]
val_files   = [all_files[i] for i in perm[n_train:]]
print(f"Train : {len(train_files)} files ({100*(1-VAL_SPLIT):.0f}%)")
print(f"Val   : {len(val_files)}   files ({100*VAL_SPLIT:.0f}%)")

class_weights = compute_class_weights(
    train_files,
    NUM_CLASSES,
    SEP,
    LABEL_COL
)
print("Class weights:", class_weights.detach().cpu().numpy().round(4).tolist())

dataset_kwargs = dict(
    num_points=NUM_POINTS, num_layers=NUM_LAYERS,
    num_neighbors=NUM_NEIGHBORS, decimation=DECIMATION,
    xyz_cols=XYZ_COLS, feat_cols=FEAT_COLS,
    label_col=LABEL_COL, sep=SEP)

train_dataset = PointCloudDataset(train_files, **dataset_kwargs)
val_dataset   = PointCloudDataset(val_files,   **dataset_kwargs)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                            shuffle=True,  num_workers=NUM_WORKERS,
                            collate_fn=collate_fn)
val_loader    = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                            shuffle=False, num_workers=NUM_WORKERS,
                            collate_fn=collate_fn)
print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")

# ── Build Purdue model (D_IN=5, NUM_CLASSES=10) ────────────────────────────
print("\nBuilding Purdue model (D_IN=5, NUM_CLASSES=10) ...")
model = RandLANet(D_IN, NUM_CLASSES, NUM_NEIGHBORS, DECIMATION, device)

# ── Stage 1: Transfer encoder/decoder from Semantic3D ─────────────────────
if RESUME_CKPT is None:
    print(f"\nLoading Semantic3D pretrained weights from '{SEMANTIC3D_CKPT.name}' ...")
    s3d_sd = torch.load(SEMANTIC3D_CKPT, map_location=device)['model_state_dict']

    # Keys to transfer: everything except fc0 (D_IN mismatch) and fc1 (NUM_CLASSES mismatch)
    skip_prefixes = ('fc0', 'fc1')
    transfer_sd   = {k: v for k, v in s3d_sd.items()
                     if not any(k.startswith(p) for p in skip_prefixes)}

    # Load transferable weights — strict=False allows mismatched fc0/fc1
    missing, unexpected = model.load_state_dict(transfer_sd, strict=False)

    print(f"  Transferred : {len(transfer_sd)} keys  (encoder + decoder + bottleneck + bn0)")
    print(f"  Missing     : {len(missing)} keys  (fc0 + fc1 — randomly initialized for Purdue)")
    print(f"  Unexpected  : {len(unexpected)} keys")

    # ── Two-stage optimizer: low LR for pretrained, high LR for new layers ──
    pretrained_params = [p for n, p in model.named_parameters()
                         if not any(n.startswith(pf) for pf in skip_prefixes)]
    new_params        = [p for n, p in model.named_parameters()
                         if any(n.startswith(pf) for pf in skip_prefixes)]

    optimizer = torch.optim.Adam([
        {'params': pretrained_params, 'lr': LR_PRETRAINED},
        {'params': new_params,        'lr': LR_NEW_LAYERS},
    ])
    print(f"\n  Pretrained params : {len(pretrained_params)}  LR={LR_PRETRAINED}")
    print(f"  New layer params  : {len(new_params)}          LR={LR_NEW_LAYERS}")
    first_epoch = 1

else:
    # Resume from a transfer learning checkpoint
    print(f"\nResuming transfer learning from '{RESUME_CKPT}' ...")
    ckpt        = torch.load(RESUME_CKPT, map_location=device)
    first_epoch = ckpt['epoch'] + 1
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer   = torch.optim.Adam(model.parameters(), lr=LR_PRETRAINED)
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    print(f"  Resuming from epoch {first_epoch}")

scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, SCHEDULER_GAMMA)
criterion = nn.CrossEntropyLoss(weight=class_weights)

# ── Evaluate ──────────────────────────────────────────────────────────────
def evaluate(model, loader, criterion, device):
    model.eval()
    losses, accuracies, ious = [], [], []
    with torch.no_grad():
        for inputs, labels in tqdm(loader, desc='Validation', leave=False):
            labels = labels.to(device)
            inputs = inputs_to_device(inputs, device)
            try:
                scores    = model(inputs)
                scores_t  = scores.permute(0, 2, 1)
                loss      = criterion(scores_t, labels)
                losses.append(loss.cpu().item())
                for b in range(scores_t.size(0)):
                    accuracies.append(accuracy(scores_t[b], labels[b]))
                    ious.append(intersection_over_union(scores_t[b], labels[b]))
            except RuntimeError as e:
                print(f"\n  [Val WARNING] Skipping batch: {e}")
                torch.cuda.empty_cache()
                continue
    return (np.mean(losses)        if losses     else 0.0,
            np.nanmean(accuracies) if accuracies else 0.0,
            np.nanmean(ious)       if ious        else 0.0)

# ── Training loop ─────────────────────────────────────────────────────────
print("\nStarting transfer learning fine-tuning ...")
print(f"  Stage: Semantic3D pretrained → Purdue fine-tune")
print(f"  Encoder/Decoder LR : {LR_PRETRAINED}")
print(f"  fc0 + fc1 LR       : {LR_NEW_LAYERS}")

with SummaryWriter(str(LOGS_DIR)) as writer:
    for epoch in range(first_epoch, EPOCHS + 1):
        print(f"\n=== EPOCH {epoch}/{EPOCHS} ===")
        t0 = time.time()
        model.train()
        losses, accuracies, ious = [], [], []

        for inputs, labels in tqdm(train_loader, desc='Training', leave=False):
            labels = labels.to(device)
            inputs = inputs_to_device(inputs, device)
            optimizer.zero_grad()
            try:
                scores    = model(inputs)
                scores_t  = scores.permute(0, 2, 1)
                loss      = criterion(scores_t, labels)
                loss.backward()
                optimizer.step()
                losses.append(loss.cpu().item())
                for b in range(scores_t.size(0)):
                    accuracies.append(accuracy(scores_t[b], labels[b]))
                    ious.append(intersection_over_union(scores_t[b], labels[b]))
            except RuntimeError as e:
                print(f"\n  [Train WARNING] Skipping batch: {e}")
                torch.cuda.empty_cache()
                continue

        scheduler.step()

        if not losses:
            print("  No batches completed — skipping metrics.")
            continue

        val_loss, val_acc, val_iou = evaluate(model, val_loader, criterion, device)

        t1      = time.time()
        d       = t1 - t0
        tr_loss = np.mean(losses)
        tr_acc  = np.nanmean([a for sub in accuracies for a in sub])
        tr_iou  = np.nanmean([i for sub in ious       for i in sub])

        print(f"  Train — Loss: {tr_loss:.5f}  Acc: {tr_acc:.4f}  mIoU: {tr_iou:.4f}")
        print(f"  Val   — Loss: {val_loss:.5f}  Acc: {val_acc:.4f}  mIoU: {val_iou:.4f}")
        print(f"  Time  — {'%.0f s' % d if d < 60 else '%.0f min %02.0f s' % divmod(d, 60)}")

        if device.type == "cuda":
            mem = torch.cuda.memory_reserved(0) / 1e9
            print(f"  GPU mem : {mem:.2f} GB")
            torch.cuda.empty_cache()

        writer.add_scalars('Loss',     {'Train': tr_loss, 'Val': val_loss}, epoch)
        writer.add_scalars('Accuracy', {'Train': tr_acc,  'Val': val_acc},  epoch)
        writer.add_scalars('mIoU',     {'Train': tr_iou,  'Val': val_iou},  epoch)

        if epoch % SAVE_FREQ == 0:
            ckpt_path = LOGS_DIR / f"transfer_checkpoint_{epoch:03d}.pth"
            torch.save({
                'epoch':                epoch,
                'model_state_dict':     model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
            }, ckpt_path)
            print(f"  Checkpoint → '{ckpt_path}'")

print("\nTransfer learning complete.")
print(f"Checkpoints saved in '{LOGS_DIR}'")

Using device : cuda:0
  GPU    : NVIDIA RTX A4000
  Memory : 17.2 GB

Total tile files found : 2
Train : 1 files (80%)
Val   : 1   files (20%)

Computing class weights from training files ...


Raw class counts: [0, 0, 0, 0, 0, 428516, 784406, 1808328, 0, 15974]
Class weights: [1.5987000465393066, 1.5987000465393066, 1.5987000465393066, 1.5987000465393066, 1.5987000465393066, 0.09719999879598618, 0.09290000051259995, 0.08749999850988388, 1.5987000465393066, 0.13019999861717224]
Train batches : 1
Val batches   : 1

Building Purdue model (D_IN=5, NUM_CLASSES=10) ...

Loading Semantic3D pretrained weights from 'randlanet_semantic3d.pth' ...
  Transferred : 312 keys  (encoder + decoder + bottleneck + bn0)
  Missing     : 16 keys  (fc0 + fc1 — randomly initialized for Purdue)
  Unexpected  : 0 keys

  Pretrained params : 186  LR=0.0001
  New layer params  : 12          LR=0.001

Starting transfer learning fine-tuning ...
  Stage: Semantic3D pretrained → Purdue fine-tune
  Encoder/Decoder LR : 0.0001
  fc0 + fc1 LR       : 0.001

=== EPOCH 1/50 ===


  Train — Loss: 2.55502  Acc: 0.0545  mIoU: 0.0133
  Val   — Loss: 4.26459  Acc: 0.1250  mIoU: 0.0000
  Time  — 27 s
  GPU mem : 0.67 GB

=== EPOCH 2/50 ===


  Train — Loss: 2.46493  Acc: 0.0628  mIoU: 0.0167
  Val   — Loss: 4.31425  Acc: 0.1339  mIoU: 0.0050
  Time  — 26 s
  GPU mem : 0.70 GB

=== EPOCH 3/50 ===


  Train — Loss: 2.44725  Acc: 0.0696  mIoU: 0.0186
  Val   — Loss: 3.71928  Acc: 0.1480  mIoU: 0.0155
  Time  — 26 s
  GPU mem : 0.70 GB

=== EPOCH 4/50 ===


...

  Train — Loss: 2.16930  Acc: 0.1596  mIoU: 0.0450
  Val   — Loss: 3.46990  Acc: 0.1515  mIoU: 0.0133
  Time  — 26 s
  GPU mem : 0.70 GB

=== EPOCH 8/50 ===


  Train — Loss: 2.16597  Acc: 0.1486  mIoU: 0.0442
  Val   — Loss: 3.33000  Acc: 0.1539  mIoU: 0.0140
  Time  — 26 s
  GPU mem : 0.70 GB

=== EPOCH 9/50 ===


  Train — Loss: 2.09612  Acc: 0.1717  mIoU: 0.0493
  Val   — Loss: 3.50176  Acc: 0.0354  mIoU: 0.0169
  Time  — 25 s
  GPU mem : 0.70 GB

=== EPOCH 10/50 ===


  Train — Loss: 2.09853  Acc: 0.1647  mIoU: 0.0491
  Val   — Loss: 2.52442  Acc: 0.1450  mIoU: 0.0117
  Time  — 25 s
  GPU mem : 0.70 GB
  Checkpoint → '/mnt/d/test/runs/transfer_learning/transfer_checkpoint_010.pth'

=== EPOCH 11/50 ===


KeyboardInterrupt: 